<div style="padding: 28px; border-radius: 18px; background: linear-gradient(120deg, #07111f 0%, #0e2940 45%, #0f766e 100%); color: #f8fafc; margin-bottom: 14px;">
  <div style="font-size: 12px; text-transform: uppercase; opacity: 0.85;">NeMo Evaluator SDK Showcase</div>
  <h1 style="margin: 8px 0 10px; font-weight: 800; line-height: 1.1;">High-Level <code style="background: rgba(255,255,255,0.16); color: #f8fafc; padding: 2px 8px; border-radius: 8px;">Evaluator.run_sync(...)</code> Walkthrough</h1>
</div>

<table>
  <tr>
    <td><b>Goal</b></td>
    <td>Show the best UX for running and inspecting offline evaluations.</td>
  </tr>
  <tr>
    <td><b>Focus</b></td>
    <td>Evaluator-first API ergonomics + result exploration utilities.</td>
  </tr>
  <tr>
    <td><b>Outputs</b></td>
    <td><code>print_summary</code>, <code>format_summary</code>, <code>to_records</code>, <code>to_table</code>, <code>to_pandas</code>.</td>
  </tr>
</table>

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import pyarrow as pa
from IPython.display import Markdown, display
from nemo_evaluator_sdk.execution.evaluator import Evaluator
from nemo_evaluator_sdk.metrics.exact_match import ExactMatchMetric
from nemo_evaluator_sdk.values import DatasetRows, EvaluationResult

## Notebook Map

- [Setup workspace](#setup-workspace)
- [Metric executor showcase](#metric-showcase)
- [Input mode 1: inline list](#inline-list)
- [Input mode 2: DatasetRows](#datasetrows)
- [Input mode 3: pyarrow.Table](#pyarrow-table)
- [Input mode 4: single file path](#file-path)
- [Input mode 5: directory + pattern](#directory-pattern)
- [Result APIs showcase](#result-apis)


<a id="setup-workspace"></a>

## Setup Workspace

We create a temporary local workspace with one standalone JSONL file and a sharded directory dataset.


In [ ]:
def write_jsonl(path: Path, rows: list[dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")


def display_df(title: str, records: list[dict]) -> None:
    display(Markdown(f"### {title}"))
    if pd is None:
        print(records)
        return
    display(pd.DataFrame.from_records(records))


def summarize_run(name: str, result: EvaluationResult) -> dict[str, object]:
    aggregate = result.to_records(view="aggregate")
    exact = next((row for row in aggregate if row.get("name") == "exact_match"), aggregate[0] if aggregate else {})
    return {
        "run": name,
        "rows": len(result.row_scores),
        "exact_match_mean": exact.get("mean"),
        "count": exact.get("count"),
        "nan_count": exact.get("nan_count"),
    }


tmp_dir = TemporaryDirectory()
workspace = Path(tmp_dir.name)

base_rows = [
    {"question": "2 + 2", "expected": "4", "prediction": "4"},
    {"question": "Largest planet", "expected": "Jupiter", "prediction": "Saturn"},
    {"question": "Primary color of the sky", "expected": "blue", "prediction": "Blue"},
]

file_path = workspace / "single-file" / "eval.jsonl"
write_jsonl(file_path, base_rows)

dataset_dir = workspace / "directory-input"
write_jsonl(dataset_dir / "part-000.jsonl", base_rows[:2])
write_jsonl(dataset_dir / "part-001.jsonl", base_rows[2:])
write_jsonl(dataset_dir / "ignored.jsonl", [{"expected": "x", "prediction": "y"}])

evaluator = Evaluator()

runs: dict[str, EvaluationResult] = {}

workspace_snapshot = [
    {"path": str(file_path), "purpose": "Single-file input"},
    {"path": str(dataset_dir / "part-000.jsonl"), "purpose": "Sharded input part 1"},
    {"path": str(dataset_dir / "part-001.jsonl"), "purpose": "Sharded input part 2"},
    {"path": str(dataset_dir / "ignored.jsonl"), "purpose": "Intentionally excluded question/answer pattern"},
]

display(Markdown(f"**Workspace:** `{workspace}`"))
display_df("Files created for this notebook", workspace_snapshot)

<a id="quickstart"></a>

## Quickstart in 30 Seconds
<a id="metric-showcase"></a>
A compact, realistic example for each built-in metric executor: create the executor, evaluate a tiny dataset, and inspect a short summary.


In [ ]:
from nemo_evaluator_sdk.metrics.bleu import BLEUMetric
from nemo_evaluator_sdk.metrics.f1 import F1Metric
from nemo_evaluator_sdk.metrics.number_check import NumberCheckMetric
from nemo_evaluator_sdk.metrics.rouge import ROUGEMetric
from nemo_evaluator_sdk.metrics.string_check import StringCheckMetric
from nemo_evaluator_sdk.metrics.tool_calling import ToolCallingMetric

In [ ]:
# ExactMatchMetric
exact_metric = ExactMatchMetric(reference="{{ expected }}", candidate="{{ prediction }}")
exact_result = evaluator.run_sync(
    metrics=exact_metric,
    dataset=[
        {"expected": "Paris", "prediction": "Paris"},
        {"expected": "Jupiter", "prediction": "Saturn"},
    ],
)

runs["Quickstart"] = exact_result

display(Markdown("### ExactMatchMetric"))
exact_result.print_summary(max_rows=2)

In [ ]:
# F1Metric
f1_metric = F1Metric(reference="{{ reference }}", candidate="{{ prediction }}")
f1_result = evaluator.run_sync(
    metrics=f1_metric,
    dataset=[
        {"reference": "The Eiffel Tower is in Paris.", "prediction": "Eiffel Tower is in Paris"},
        {"reference": "Python is a programming language.", "prediction": "Python is a snake"},
    ],
)
display(Markdown("### F1Metric"))
f1_result.print_summary(max_rows=2)

In [ ]:
# BLEUMetric
bleu_metric = BLEUMetric(references=["{{ reference }}"], candidate="{{ prediction }}")
bleu_result = evaluator.run_sync(
    metrics=bleu_metric,
    dataset=[
        {"reference": "the cat is on the mat", "prediction": "the cat is on the mat"},
        {"reference": "a dog runs in the park", "prediction": "dog runs at park"},
    ],
)
display(Markdown("### BLEUMetric"))
bleu_result.print_summary(max_rows=2)

In [ ]:
# NumberCheckMetric
number_metric = NumberCheckMetric(
    operation="absolute difference",
    left_template="{{ expected_value }}",
    right_template="{{ model_value }}",
    epsilon=0.01,
)
number_result = evaluator.run_sync(
    metrics=number_metric,
    dataset=[
        {"expected_value": "3.1416", "model_value": "3.1410"},
        {"expected_value": "2.50", "model_value": "2.61"},
    ],
)
display(Markdown("### NumberCheckMetric"))
number_result.print_summary(max_rows=2)

In [ ]:
# ROUGEMetric
rouge_metric = ROUGEMetric(reference="{{ reference_summary }}", candidate="{{ model_summary }}")
rouge_result = evaluator.run_sync(
    metrics=rouge_metric,
    dataset=[
        {
            "reference_summary": "The launch succeeded after two delays caused by weather.",
            "model_summary": "After weather delays, the launch was successful.",
        },
        {
            "reference_summary": "Revenue increased 12 percent year over year.",
            "model_summary": "The company reported lower yearly revenue.",
        },
    ],
)
display(Markdown("### ROUGEMetric"))
rouge_result.print_summary(max_rows=2)

In [ ]:
# StringCheckMetric
string_metric = StringCheckMetric(
    operation="contains",
    left_template="{{ answer_text }}",
    right_template="{{ required_phrase }}",
)
string_result = evaluator.run_sync(
    metrics=string_metric,
    dataset=[
        {"answer_text": "The SLA is 99.9% uptime.", "required_phrase": "99.9%"},
        {"answer_text": "Support is available weekdays only.", "required_phrase": "24/7"},
    ],
)
display(Markdown("### StringCheckMetric"))
string_result.print_summary(max_rows=2)

In [ ]:
# ToolCallingMetric


def _tool_response(tool_calls: list[dict]) -> dict:
    return {"choices": [{"message": {"tool_calls": tool_calls}}]}


tool_metric = ToolCallingMetric(reference="{{item.reference}}")
tool_result = evaluator.run_sync(
    metrics=tool_metric,
    dataset=[
        {
            "reference": [{"function": {"name": "sum", "arguments": {"x": 1, "y": 2}}}],
            "response": _tool_response([{"function": {"name": "sum", "arguments": '{"x": 1, "y": 2}'}}]),
        },
        {
            "reference": [{"function": {"name": "weather_get_forecast", "arguments": {"city": "Paris"}}}],
            "response": _tool_response(
                [{"function": {"name": "weather_get_forecast", "arguments": '{"city": "London"}'}}]
            ),
        },
    ],
)
display(Markdown("### ToolCallingMetric"))
tool_result.print_summary()

<a id="inline-list"></a>

## Input Mode 1: Inline `list[dict[str, Any]]`

Best for lightweight experiments and rapid iteration.


In [ ]:
inline_rows = [
    {"expected": "blue", "model_output": "Blue", "prompt": "Primary color of the sky"},
    {"expected": "Jupiter", "model_output": "Saturn", "prompt": "Largest planet"},
    {"expected": "Paris", "model_output": "Paris", "prompt": "Capital of France"},
]

metric = ExactMatchMetric(reference="{{ expected }}", candidate="{{ model_output }}")
inline_result = evaluator.run_sync(metrics=metric, dataset=inline_rows)

runs["Inline list[dict]"] = inline_result

# use pandas dataframe to display the results
display_df("Inline aggregate view", inline_result.to_records(view="aggregate"))
display_df("Inline rows view", inline_result.to_records(view="rows"))

<a id="datasetrows"></a>

## Input Mode 2: `DatasetRows`

Use this typed wrapper when you want schema-validated inline rows.


In [ ]:
dataset_rows = DatasetRows(
    rows=[
        {"expected": "cat", "model_output": "cat", "prompt": "Name this animal"},
        {"expected": "dog", "model_output": "wolf", "prompt": "Name this pet"},
    ]
)

metric = ExactMatchMetric(reference="{{ expected }}", candidate="{{ model_output }}")
dataset_rows_result = evaluator.run_sync(metrics=metric, dataset=dataset_rows)

runs["DatasetRows"] = dataset_rows_result

dataset_rows_result.print_summary(max_rows=2)

<a id="pyarrow-table"></a>

## Input Mode 3: `pyarrow.Table`

Ideal when your pipeline is Arrow-native and you want zero conversion overhead in user code.


In [ ]:
arrow_table = pa.Table.from_pylist(base_rows)

metric = ExactMatchMetric(reference="{{ expected }}", candidate="{{ prediction }}")
arrow_result = evaluator.run_sync(metrics=metric, dataset=arrow_table)

runs["pyarrow.Table"] = arrow_result

arrow_result.print_summary(max_rows=2)

<a id="file-path"></a>

## Input Mode 4: Local File Path

Use a path directly when data is already materialized on disk.


In [ ]:
metric = ExactMatchMetric(reference="{{ expected }}", candidate="{{ prediction }}")
file_result = evaluator.run_sync(metrics=metric, dataset=file_path)

runs["File path"] = file_result

file_result.print_summary(max_rows=2)

<a id="directory-pattern"></a>

## Input Mode 5: Directory Path + `pattern`

For sharded datasets, pass the directory and target only matching files.


In [ ]:
metric = ExactMatchMetric(reference="{{ expected }}", candidate="{{ prediction }}")
directory_result = evaluator.run_sync(metrics=metric, dataset=dataset_dir, pattern="part-*.jsonl")

runs["Directory + pattern"] = directory_result

directory_result.print_summary(max_rows=2)

<a id="result-apis"></a>

## Result APIs

All runs return `EvaluationResult` with the same helper surface.


In [ ]:
from pprint import pprint

result = runs["Inline list[dict]"]

print("type:", type(result))
print("\nFirst row score payload:")
pprint(result.row_scores[0].model_dump())

row_records = result.to_records(view="rows")
aggregate_records = result.to_records(view="aggregate")

rows_table = result.to_table(view="rows")
aggregate_table = result.to_table(view="aggregate")

print("\nRows table schema:")
print(rows_table.schema)

print("\nFormatted summary:")
print(result.format_summary(max_rows=2))

if pd is not None:
    display_df("to_pandas(rows)", result.to_pandas(view="rows").to_dict(orient="records"))
    display_df("to_pandas(aggregate)", result.to_pandas(view="aggregate").to_dict(orient="records"))
else:
    print("\nInstall nemo-evaluator-sdk[pandas] for DataFrame helpers.")

## Takeaways

- `evaluator.run_sync(metrics=..., dataset=...)` gives one clean API across inline rows, Arrow data, and local files.
- Templates (`reference`, `candidate`) handle the mapping between dataset columns and metric inputs.
- `EvaluationResult` supports both human-friendly summaries and dataframe/table pipelines.
- The exact same workflow scales from local notebook demos to production data slices.